# Linear Regression — Exercises

Companion to the note [Linear Regression](Linear%20Regression.md).

Practise the normal equations, gradient descent, ridge, basis functions, the probabilistic (MLE) view and regression metrics, first by hand and then from scratch in NumPy.

**14 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
rng = np.random.default_rng(42)
# A small shared dataset: y = 3 + 2x + noise
x_toy = np.array([0., 1., 2., 3., 4.])
y_toy = np.array([3.1, 4.9, 7.2, 8.8, 11.0])

## Part A · Concepts

### Exercise 1 · What does 'linear' mean?
*🧠 Concept · ★☆☆*

Which of these models are *linear regression* models (linear in the parameters)? Explain briefly.

1. $\hat y = w_0 + w_1 x + w_2 x^2$
2. $\hat y = w_0 + w_1 \sin(x)$
3. $\hat y = w_0 + e^{w_1 x}$
4. $\hat y = w_0 + w_1 x_1 x_2$

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: is $\hat y$ a weighted sum of fixed functions $\phi_j(x)$ with the weights as coefficients?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1, 2 and 4 are linear in $w$: they are $w^\top\phi(x)$ with $\phi = (1, x, x^2)$, $(1,\sin x)$, $(1, x_1x_2)$.
3 is **not**: $w_1$ sits inside the exponential, so the model is non-linear in the parameters and has no closed-form least-squares solution.

</details>

### Exercise 2 · Assumptions and residual plots
*🧠 Concept · ★☆☆*

You fit a linear regression and plot residuals $r_i = y_i - \hat y_i$ against $\hat y_i$.
Which assumption is violated in each case?

(a) The residuals form a U-shape.  (b) The spread of residuals grows with $\hat y$ (a funnel).  (c) Residuals of consecutive time steps are strongly correlated.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The note lists four assumptions: linearity, independent errors, constant variance, no strong multicollinearity.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Linearity**: a systematic pattern means the mean of $y$ is not linear in the features; add basis functions.
(b) **Homoscedasticity** (constant variance) is violated; try transforming $y$ (e.g. $\log y$) or weighted least squares.
(c) **Independent errors** is violated (autocorrelation), typical for time series.

</details>

### Exercise 3 · MLE ⇔ least squares
*🧠 Concept · ★★☆*

Assume $y_i = w^\top x_i + \varepsilon_i$ with $\varepsilon_i \sim \mathcal N(0,\sigma^2)$ i.i.d.
Write the log-likelihood and show that maximising it over $w$ gives the same solution as minimising the SSE.
What would change if the noise were Laplace distributed instead?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Take the log of a product of Gaussian densities; terms not involving $w$ do not matter for the argmax.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(y\mid X,w) = \sum_i \left[-\tfrac12\log(2\pi\sigma^2) - \tfrac{(y_i - w^\top x_i)^2}{2\sigma^2}\right]
= \text{const} - \tfrac{1}{2\sigma^2}\,\text{SSE}(w)$.
Maximising this is the same as minimising SSE. With Laplace noise, $p(\varepsilon) \propto e^{-|\varepsilon|/b}$, the MLE minimises
$\sum_i |y_i - w^\top x_i|$ (least absolute deviations), which is more robust to outliers.

</details>

## Part B · By hand

### Exercise 4 · Simple regression by hand
*✍️ By hand · ★☆☆*

For the toy data $x = (0,1,2,3,4)$, $y = (3.1, 4.9, 7.2, 8.8, 11.0)$, compute the least-squares line
$\hat y = b + w x$ using
$w = \frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}$, $b = \bar y - w\bar x$.
Then store your numbers below.

In [ ]:
w_hand = None  # slope
b_hand = None  # intercept

check('slope', w_hand, np.polyfit(x_toy, y_toy, 1)[0], tol=1e-2)
check('intercept', b_hand, np.polyfit(x_toy, y_toy, 1)[1], tol=1e-2)

<details>
<summary><b>💡 Hint 1</b></summary>

$\bar x = 2$, $\bar y = 7$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Deviations $x_i-\bar x = (-2,-1,0,1,2)$, so $\sum(x_i-\bar x)^2 = 10$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\bar x = 2,\ \bar y = 7$. $\sum (x_i-\bar x)(y_i-\bar y) = (-2)(-3.9)+(-1)(-2.1)+0+1(1.8)+2(4.0) = 7.8+2.1+1.8+8.0 = 19.7$.
$w = 19.7/10 = 1.97$, $b = 7 - 1.97\cdot 2 = 3.06$.

```python
w_hand = 1.97
b_hand = 3.06
```

</details>

### Exercise 5 · Normal equations in matrix form
*✍️ By hand · ★★☆*

Derive the normal equations: starting from $L(w) = \lVert Xw - y\rVert^2$ (rows of $X$ are samples, with a column of ones),
compute $\nabla_w L$, set it to zero and solve for $w$. When does $X^\top X$ fail to be invertible?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Expand $L = w^\top X^\top X w - 2 y^\top X w + y^\top y$ and use $\nabla_w (w^\top A w) = 2Aw$ for symmetric $A$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_w L = 2X^\top X w - 2X^\top y = 0 \Rightarrow X^\top X\, w = X^\top y \Rightarrow w = (X^\top X)^{-1} X^\top y$.
$X^\top X$ is singular when the columns of $X$ are linearly dependent: perfectly collinear features, a duplicated column,
or fewer samples than features ($N < d$). Ridge's $\lambda I$ fixes this.

</details>

### Exercise 6 · Gradient of the MSE
*✍️ By hand · ★☆☆*

For $L(w) = \frac1N\lVert Xw-y\rVert^2$ the note gives $\nabla L = \frac2N X^\top(Xw-y)$.
With $X = \begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$, $y = (1,2,2)^\top$ and $w = (0,0)^\top$, compute the gradient by hand.

In [ ]:
grad_hand = None  # e.g. [g0, g1]

_X = np.array([[1,0],[1,1],[1,2.]]); _y = np.array([1,2,2.])
check('gradient', grad_hand, 2/3*_X.T@(_X@np.zeros(2)-_y))

<details>
<summary><b>💡 Hint</b></summary>

$Xw - y = -y$ when $w=0$. Then multiply by $X^\top$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Xw - y = (-1,-2,-2)^\top$. $X^\top(Xw-y) = (-1-2-2,\ 0-2-4) = (-5,-6)$. Times $\frac23$: $\nabla L = (-10/3, -4) \approx (-3.333, -4)$.

```python
grad_hand = [-10/3, -4]
```

</details>

### Exercise 7 · Ridge closed form
*✍️ By hand · ★★☆*

Show that minimising $\lVert Xw - y\rVert^2 + \lambda \lVert w\rVert^2$ gives $w = (X^\top X + \lambda I)^{-1} X^\top y$,
and argue why $X^\top X + \lambda I$ is always invertible for $\lambda>0$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$X^\top X$ is positive semi-definite: its eigenvalues are $\ge 0$. What does adding $\lambda I$ do to the eigenvalues?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Gradient: $2X^\top(Xw-y) + 2\lambda w = 0 \Rightarrow (X^\top X + \lambda I)w = X^\top y$.
$X^\top X$ is PSD (eigenvalues $\mu_j \ge 0$); adding $\lambda I$ shifts every eigenvalue to $\mu_j + \lambda > 0$, so the matrix is positive definite and invertible.
(In practice the intercept is usually not penalised.)

</details>

## Part C · Code from scratch

### Exercise 8 · Normal equations in NumPy
*💻 Code · ★☆☆*

Implement `fit_normal(X, y)` that adds a bias column and returns $w$ (bias first) via the normal equations.
Use `np.linalg.solve` rather than an explicit inverse.

In [ ]:
def fit_normal(X, y):
    # TODO: add a column of ones, solve (X^T X) w = X^T y
    return None

X_lin = rng.normal(size=(100, 3))
y_lin = 1.5 + X_lin @ np.array([2.0, -1.0, 0.5]) + 0.1 * rng.normal(size=100)
w_ne = fit_normal(X_lin, y_lin)

_ref = LinearRegression().fit(X_lin, y_lin)
check('normal equations', w_ne, np.r_[_ref.intercept_, _ref.coef_])

<details>
<summary><b>💡 Hint</b></summary>

`Xb = np.c_[np.ones(len(X)), X]`; then `np.linalg.solve(Xb.T @ Xb, Xb.T @ y)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`solve` is faster and numerically more stable than `inv`. For ill-conditioned problems `np.linalg.lstsq` (SVD-based) is even safer.

```python
def fit_normal(X, y):
    Xb = np.c_[np.ones(len(X)), X]
    return np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

X_lin = rng.normal(size=(100, 3))
y_lin = 1.5 + X_lin @ np.array([2.0, -1.0, 0.5]) + 0.1 * rng.normal(size=100)
w_ne = fit_normal(X_lin, y_lin)
```

</details>

### Exercise 9 · Batch gradient descent
*💻 Code · ★★☆*

Implement `fit_gd(X, y, lr, epochs)` for the MSE loss (with bias column) and run it on `X_lin, y_lin`.
It should reach the normal-equation solution to about 3 decimals. Then try `lr=1.5`: what happens and why?

In [ ]:
def fit_gd(X, y, lr=0.1, epochs=500):
    # TODO: w = zeros; repeat: w -= lr * (2/N) X^T (Xw - y)
    return None

w_gd = fit_gd(X_lin, y_lin)

check('gradient descent', w_gd, np.r_[LinearRegression().fit(X_lin, y_lin).intercept_, LinearRegression().fit(X_lin, y_lin).coef_], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Use the gradient from Exercise 6. If it diverges, the step size exceeds $2/\lambda_{\max}$ of the Hessian $\frac2N X^\top X$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With `lr=1.5` the iterates blow up: GD on a quadratic is stable only for $\eta < 2/\lambda_{\max}(H)$ with $H=\frac2N X^\top X$
(here $\lambda_{\max}\approx 2.2$, so the limit is about $0.9$). See [[Gradient Descent]].

```python
def fit_gd(X, y, lr=0.1, epochs=500):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * 2 / len(y) * Xb.T @ (Xb @ w - y)
    return w

w_gd = fit_gd(X_lin, y_lin)
```

</details>

### Exercise 10 · Ridge from scratch
*💻 Code · ★★☆*

Implement `fit_ridge(X, y, lam)` **without penalising the intercept**: centre $X$ and $y$, solve the ridge system for the
slopes, then recover the intercept. Compare with `sklearn.linear_model.Ridge(alpha=lam)`.

In [ ]:
def fit_ridge(X, y, lam):
    # TODO: return np.r_[intercept, slopes]
    return None

w_ridge = fit_ridge(X_lin, y_lin, lam=10.0)

_r = Ridge(alpha=10.0).fit(X_lin, y_lin)
check('ridge', w_ridge, np.r_[_r.intercept_, _r.coef_])

<details>
<summary><b>💡 Hint</b></summary>

After centring, the intercept is $b = \bar y - \bar x^\top w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Centring removes the intercept from the problem, so the penalty only acts on the slopes, exactly what sklearn does.

```python
def fit_ridge(X, y, lam):
    xm, ym = X.mean(0), y.mean()
    Xc, yc = X - xm, y - ym
    w = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ yc)
    return np.r_[ym - xm @ w, w]

w_ridge = fit_ridge(X_lin, y_lin, lam=10.0)
```

</details>

### Exercise 11 · Polynomial basis functions
*💻 Code · ★★☆*

Generate $x\in[0,1]$ (20 points) and $y = \sin(2\pi x) + \varepsilon$, $\varepsilon\sim\mathcal N(0, 0.2^2)$.
Write `poly_features(x, degree)` and fit degrees 1, 3 and 9 with your `fit_normal`-style solver (use `lstsq`).
Report training RMSE for each degree in `rmse_by_degree` (a dict). Which degree would you trust on new data?

In [ ]:
def poly_features(x, degree):
    # TODO: return an (N, degree) matrix with columns x, x^2, ..., x^degree (bias added by the solver)
    return None

x_s = np.linspace(0, 1, 20)
y_s = np.sin(2 * np.pi * x_s) + 0.2 * rng.normal(size=20)
rmse_by_degree = None  # {1: ..., 3: ..., 9: ...}

if rmse_by_degree is not None:
    check('training RMSE falls with degree', rmse_by_degree[1] > rmse_by_degree[3] > rmse_by_degree[9], True)
else:
    check('training RMSE falls with degree', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.vander(x, degree + 1, increasing=True)[:, 1:]` gives $x, x^2, \dots$. Training error always drops as capacity grows.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training RMSE decreases monotonically with degree, but degree 9 interpolates the noise and oscillates between points.
Degree 3 captures one period of a sine well and is what you would trust; confirm with a validation set or
[[Cross-Validation and Model Selection]]. This is the bias–variance tradeoff in action ([[Bias-Variance Tradeoff]]).

```python
def poly_features(x, degree):
    return np.vander(x, degree + 1, increasing=True)[:, 1:]

x_s = np.linspace(0, 1, 20)
y_s = np.sin(2 * np.pi * x_s) + 0.2 * rng.normal(size=20)
rmse_by_degree = {}
for d in (1, 3, 9):
    Phi = np.c_[np.ones(20), poly_features(x_s, d)]
    w = np.linalg.lstsq(Phi, y_s, rcond=None)[0]
    rmse_by_degree[d] = np.sqrt(np.mean((Phi @ w - y_s) ** 2))
print(rmse_by_degree)
```

</details>

### Exercise 12 · Metrics: RMSE, MAE, R²
*💻 Code · ★☆☆*

Implement `rmse`, `mae` and `r2` from their definitions and evaluate the toy line from Exercise 4 ($\hat y = 3.06 + 1.97x$).

In [ ]:
def rmse(y, yhat):
    return None
def mae(y, yhat):
    return None
def r2(y, yhat):
    return None

yhat_toy = 3.06 + 1.97 * x_toy

check('R^2', r2(y_toy, yhat_toy), r2_score(y_toy, yhat_toy))
check('RMSE', rmse(y_toy, yhat_toy), np.sqrt(np.mean((y_toy-yhat_toy)**2)))

<details>
<summary><b>💡 Hint</b></summary>

$R^2 = 1 - \frac{\sum (y_i-\hat y_i)^2}{\sum (y_i - \bar y)^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$R^2 \approx 0.997$: the line explains almost all variance. RMSE punishes large errors more than MAE because errors are squared.

```python
def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))
def mae(y, yhat):
    return np.mean(np.abs(y - yhat))
def r2(y, yhat):
    return 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)

yhat_toy = 3.06 + 1.97 * x_toy
```

</details>

### Exercise 13 · Multicollinearity
*💻 Code · ★★★*

Create two almost identical features `x1` and `x2 = x1 + 1e-3*noise` and target `y = x1 + noise`.
Fit OLS on 5 different noise draws and print the coefficients. Then do the same with ridge ($\lambda=1$).
Store the standard deviation (across draws) of the first OLS coefficient in `ols_coef_std` and of the first ridge coefficient in `ridge_coef_std`.
What do you observe, and why?

In [ ]:
ols_coef_std = None
ridge_coef_std = None

if ols_coef_std is not None:
    check('ridge is far more stable than OLS', ridge_coef_std < ols_coef_std / 10, True)
else:
    check('ridge is far more stable than OLS', None, True)

<details>
<summary><b>💡 Hint</b></summary>

With collinear columns $X^\top X$ is nearly singular; tiny changes in data cause huge swings in $w$ (only $w_1+w_2$ is well determined).

</details>

<details>
<summary><b>✅ Solution</b></summary>

OLS coefficients swing wildly between draws (e.g. $-11$ and $+12$), although their sum stays close to 1. Ridge splits the weight roughly
evenly (about 0.5 each) and is stable, because $\lambda I$ removes the near-zero eigenvalue of $X^\top X$.

```python
ols, rid = [], []
for seed in range(5):
    r = np.random.default_rng(seed)
    x1 = r.normal(size=50); x2 = x1 + 1e-3 * r.normal(size=50)
    X = np.c_[x1, x2]; y = x1 + 0.1 * r.normal(size=50)
    ols.append(LinearRegression().fit(X, y).coef_)
    rid.append(Ridge(alpha=1.0).fit(X, y).coef_)
print('OLS  ', np.round(ols, 2).tolist())
print('Ridge', np.round(rid, 2).tolist())
ols_coef_std = np.std([c[0] for c in ols])
ridge_coef_std = np.std([c[0] for c in rid])
```

</details>

### Exercise 14 · Columns-as-samples form
*✍️ By hand · ★★★*

The note also gives $W = T X^\top (X X^\top)^{-1}$ when **columns** of $X$ are samples and $T$ holds the targets as columns.
Show this is the same solution as $w = (X^\top X)^{-1} X^\top y$ written in the other convention.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Let $\tilde X = X^\top$ (rows = samples) and $\tilde y = T^\top$. Transpose the rows-as-samples solution.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In rows-as-samples form, $\tilde W = (\tilde X^\top \tilde X)^{-1} \tilde X^\top \tilde y = (X X^\top)^{-1} X T^\top$.
The weight matrix in the other convention is its transpose: $W = \tilde W^\top = T X^\top (X X^\top)^{-1}$,
using that $(XX^\top)^{-1}$ is symmetric. ✔

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Linear Regression](Linear%20Regression.md).*